# telecom-pipeline data in pandas
Data: `network_snapshots` from telecom-pipeline (39 rows, 3 Spanish networks, 19 Aug – 17 Oct 2026).
Times are stored without a time zone; they appear to be UTC because the container clock is UTC.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

# Which .env file does python-dotenv find? (full path)
print("env file:", find_dotenv())

# Was the variable already set BEFORE reading .env? (e.g. by VS Code or Windows)
print("before:", "TELECOM_DB_URL" in os.environ)

# override=True: the .env value replaces any old value already in memory
load_dotenv(override=True)

# Show only the part after the @ (host, port, database), never the password
print("address:", os.environ["TELECOM_DB_URL"].split("@")[-1])


env file: c:\Users\alexp\dev\netsentinel\.env
before: True
address: localhost:5433/telecom


In [2]:
# Libraries
import os                               # access environment variables
import pandas as pd                     # tables in Python
from dotenv import load_dotenv          # reads the .env file
from sqlalchemy import create_engine    # makes the database connection

# Read .env (it searches upward from this folder until it finds the file)
load_dotenv(override=True)  # override=True: the .env value replaces any old value already in memory

# Get the connection string. os.environ works like a Java Map<String, String>
# that holds environment variables; [] fails loudly if the name is missing
db_url = os.environ["TELECOM_DB_URL"]

# The engine is a reusable "connection factory", like a JDBC DataSource in Java
engine = create_engine(db_url)

# Run the SQL query and put the result straight into a DataFrame
df = pd.read_sql("SELECT * FROM network_snapshots", engine)

# Show the first 5 rows to check it worked
df.head()

,id,asn,operator_name,prefix_count,fetched_at
0,1,AS3352,Telefonica de Espana,228,2026-08-19 11:53:32.506050
1,2,AS12430,Vodafone Spain,47,2026-08-19 11:53:33.015873
2,3,AS12479,Orange Spain,8560,2026-08-19 11:53:34.710029
3,4,AS3352,Telefonica de Espana,228,2026-08-19 12:07:39.977621
4,5,AS12430,Vodafone Spain,47,2026-08-19 12:07:40.474595


In [3]:
df.shape

(39, 5)

In [4]:
df.dtypes

id                        int64
asn                         str
operator_name               str
prefix_count              int64
fetched_at       datetime64[us]
dtype: object

In [5]:
df.isna().sum()

id               0
asn              0
operator_name    0
prefix_count     0
fetched_at       0
dtype: int64

In [6]:
df["asn"].value_counts()

asn
AS3352     13
AS12430    13
AS12479    13
Name: count, dtype: int64

In [7]:
df["run_time"] = df["fetched_at"].dt.floor("min")

In [8]:
df["run_time"].value_counts()

run_time
2026-10-03 20:50:00    6
2026-08-19 11:53:00    3
2026-08-19 12:07:00    3
2026-08-19 12:10:00    3
2026-08-19 12:13:00    3
2026-08-19 12:16:00    3
2026-08-19 12:19:00    3
2026-08-19 14:23:00    3
2026-08-19 14:25:00    3
2026-10-03 20:55:00    3
2026-10-04 15:33:00    3
2026-10-17 21:24:00    3
Name: count, dtype: int64

**Q1:** 13 runs in total, each with all 3 networks (no partial failures). 19 Aug had 7 runs and 3 Oct had 3 (two in the same minute, so rounding to the minute merged them; the raw seconds showed they were separate runs). Other dates had 1 run each.

In [9]:
sorted(df["run_time"].unique())

[Timestamp('2026-08-19 11:53:00'),
 Timestamp('2026-08-19 12:07:00'),
 Timestamp('2026-08-19 12:10:00'),
 Timestamp('2026-08-19 12:13:00'),
 Timestamp('2026-08-19 12:16:00'),
 Timestamp('2026-08-19 12:19:00'),
 Timestamp('2026-08-19 14:23:00'),
 Timestamp('2026-08-19 14:25:00'),
 Timestamp('2026-10-03 20:50:00'),
 Timestamp('2026-10-03 20:55:00'),
 Timestamp('2026-10-04 15:33:00'),
 Timestamp('2026-10-17 21:24:00')]

In [10]:
df[ df["run_time"] == "2026-10-03 20:50" ]

,id,asn,operator_name,prefix_count,fetched_at,run_time
24,25,AS3352,Telefonica de Espana,226,2026-10-03 20:50:02.267993,2026-10-03 20:50:00
25,26,AS12430,Vodafone Spain,47,2026-10-03 20:50:02.396538,2026-10-03 20:50:00
26,27,AS12479,Orange Spain,8611,2026-10-03 20:50:02.679933,2026-10-03 20:50:00
27,28,AS3352,Telefonica de Espana,226,2026-10-03 20:50:30.171348,2026-10-03 20:50:00
28,29,AS12430,Vodafone Spain,47,2026-10-03 20:50:30.361640,2026-10-03 20:50:00
29,30,AS12479,Orange Spain,8611,2026-10-03 20:50:36.896507,2026-10-03 20:50:00


In [11]:
df["run_time"].max()

Timestamp('2026-10-17 21:24:00')

In [12]:
latest_run = df["run_time"].max()

In [13]:
df[ df["run_time"] == latest_run ]

,id,asn,operator_name,prefix_count,fetched_at,run_time
36,37,AS3352,Telefonica de Espana,226,2026-10-17 21:24:26.972422,2026-10-17 21:24:00
37,38,AS12430,Vodafone Spain,47,2026-10-17 21:24:27.689891,2026-10-17 21:24:00
38,39,AS12479,Orange Spain,8354,2026-10-17 21:24:28.143550,2026-10-17 21:24:00


In [14]:
latest = df[ df["run_time"] == latest_run ]

In [15]:
latest.sort_values("prefix_count", ascending=False)

,id,asn,operator_name,prefix_count,fetched_at,run_time
38,39,AS12479,Orange Spain,8354,2026-10-17 21:24:28.143550,2026-10-17 21:24:00
36,37,AS3352,Telefonica de Espana,226,2026-10-17 21:24:26.972422,2026-10-17 21:24:00
37,38,AS12430,Vodafone Spain,47,2026-10-17 21:24:27.689891,2026-10-17 21:24:00


**Q2:** In the latest run (17 Oct 2026, 21:24 UTC), Orange Spain (AS12479) announced the most prefixes: 8,354.

In [16]:

df.groupby("asn")["prefix_count"].agg(["min", "max", "mean"])

,min,max,mean
asn,,,
AS12430,47,47,47.000000
AS12479,8354,8611,8559.846154
AS3352,226,228,227.230769


In [17]:
df_sorted = df.sort_values("fetched_at")

In [18]:
change = df_sorted.groupby("asn")["prefix_count"].agg(["first", "last"])

**Q3:** Orange varies most (8,354–8,611, a range of 257, about 3%). Telefónica barely moves (226–228). Vodafone never changes (47), so its normal variation is zero; a z-score would divide by zero, and it needs other signals.

In [19]:
change["diff"] = change["last"] - change["first"]

In [20]:
change["pct"] = change["diff"] / change["first"] * 100

In [21]:
change["pct"] = change["pct"].round(1)


In [22]:
change

,first,last,diff,pct
asn,,,,
AS12430,47,47,0,0.0
AS12479,8560,8354,-206,-2.4
AS3352,228,226,-2,-0.9



**Q4:** From first to last run, Orange dropped by 206 prefixes (−2.4%), Telefónica by 2 (−0.9%), Vodafone by 0. This is smaller than Orange's max − min (257) because first → last ignores the peak in between.

In [23]:
# 1. Jump = each run's prefix_count minus the previous run of the SAME network
#    Stored as a new column, so every jump stays next to its asn and date
df_sorted["jump"] = df_sorted.groupby("asn")["prefix_count"].diff()

# 2. Also store WHEN the previous run was: .shift() moves each value one row down,
#    again inside each network's group, so every row knows its previous run's time
df_sorted["prev_time"] = df_sorted.groupby("asn")["fetched_at"].shift()

# 3. Find the row label of the biggest jump in EITHER direction
#    .abs() turns -257 into 257; .idxmax() returns the label of the largest value
#    (NaN rows are skipped automatically)
biggest = df_sorted["jump"].abs().idxmax()

# 4. Show that row: network, its jump, and the two run times
df_sorted.loc[biggest, ["asn", "operator_name", "prev_time", "fetched_at", "jump"]]

asn                                 AS12479
operator_name                  Orange Spain
prev_time        2026-10-04 15:33:16.069196
fetched_at       2026-10-17 21:24:28.143550
jump                                 -257.0
Name: 38, dtype: object

**Q5:** The biggest jump between consecutive runs was Orange: −257 prefixes between 4 and 17 Oct 2026. The 13-day gap means we can't tell when or how fast it happened, which is why NetSentinel polls on a fixed schedule (ADR-004).